# E7 — PlotQA + InfoVQA full panel (Phase 1 P0 v3 chart-stack)

Top-to-bottom reproducer for `docs/insights/E7-plotqa-infovqa-evidence.md`.

Source per-cell CSVs (gitignored, regenerated by `scripts/analyze_e5e_wrong_correct.py`):
- `docs/insights/_data/experiment_e7_plotqa_full_per_cell.csv`
- `docs/insights/_data/experiment_e7_infographicvqa_full_per_cell.csv`

## Headline

**Anti-scaling within Gemma3 is dataset-dependent**, not architecture-bound. PlotQA shows 4B > 27B; InfoVQA reverses.
**InternVL3-8b is most robust on PlotQA but its H2 wrong > correct asymmetry collapses** — panel-side analogue of the
thinking-mode H2 collapse on MathVista. **Anchor presence improves em on 6/7 PlotQA models** (free-lunch baseline that motivates
§7.4.5 E6 mitigation).

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
pd.set_option('display.precision', 3)

## 1. (Re)build per-cell CSVs

`analyze_e5e_wrong_correct.py` walks `outputs/experiment_e7_<dataset>_full/<model>/<run>/predictions.jsonl`
and emits one CSV per dataset under `docs/insights/_data/`.

In [ ]:
import subprocess
for ds in ['experiment_e7_plotqa_full', 'experiment_e7_infographicvqa_full']:
    subprocess.run(['uv', 'run', 'python', str(ROOT / 'scripts' / 'analyze_e5e_wrong_correct.py'),
                    '--exp-dir', ds], cwd=ROOT, check=True)

plot = pd.read_csv(ROOT / 'docs/insights/_data/experiment_e7_plotqa_full_per_cell.csv')
info = pd.read_csv(ROOT / 'docs/insights/_data/experiment_e7_infographicvqa_full_per_cell.csv')
print(f'plot: {len(plot)} rows, {plot.model.nunique()} models')
print(f'info: {len(info)} rows, {info.model.nunique()} models')

## 2. Cross-model susceptibility ranking (df wrong-base, S1 anchor)

Sorted by `df(a) wrong` per dataset.

In [ ]:
def rank_panel(df, name):
    a = df[(df.cond_class == 'a') & (df.base_correct == False)][['model', 'direction_follow_M2', 'adopt_M2', 'exact_match', 'n']]
    a = a.rename(columns={'direction_follow_M2': 'df_wrong', 'adopt_M2': 'adopt_wrong', 'exact_match': 'em_wrong'})
    ac = df[(df.cond_class == 'a') & (df.base_correct == True)][['model', 'direction_follow_M2']]
    ac = ac.rename(columns={'direction_follow_M2': 'df_correct'})
    out = a.merge(ac, on='model').sort_values('df_wrong', ascending=False)
    out['gap'] = out['df_wrong'] - out['df_correct']
    print(f'=== {name} ===')
    print(out.round(3).to_string(index=False))
    print()

rank_panel(plot, 'PlotQA (n=5000)')
rank_panel(info, 'InfoVQA (n=1147)')

## 3. Anti-scaling within Gemma3 — dataset-bound

PlotQA shows 4B > 27B (anti-scaling). InfoVQA reverses.

In [ ]:
def family_scaling(df, family_models, name):
    a = df[(df.cond_class == 'a') & (df.base_correct == False) & df.model.isin(family_models)]
    print(f'{name}: {dict(zip(a.model, a.direction_follow_M2.round(3)))}')

GEMMA = ['gemma3-4b-it', 'gemma3-27b-it']
QWEN = ['qwen2.5-vl-7b-instruct', 'qwen2.5-vl-32b-instruct']
for label, df in [('PlotQA', plot), ('InfoVQA', info)]:
    print(f'-- {label} --')
    family_scaling(df, GEMMA, '  Gemma3 4B vs 27B')
    family_scaling(df, QWEN, '  Qwen2.5-VL 7B vs 32B')

## 4. Digit-pixel causality — (a − m) gap on df, wrong-base

In [ ]:
def digit_pixel_gap(df, name):
    am = df[(df.cond_class.isin(['a', 'm'])) & (df.base_correct == False)].pivot_table(
        index='model', columns='cond_class', values='direction_follow_M2', aggfunc='first')
    am['a_minus_m'] = am['a'] - am['m']
    print(f'=== {name} ===')
    print(am.sort_values('a_minus_m', ascending=False).round(3).to_string())
    print()

digit_pixel_gap(plot, 'PlotQA')
digit_pixel_gap(info, 'InfoVQA')

## 5. Anchor sometimes improves em — PlotQA 'free-lunch' pattern

All-base em(a) − em(b) per model. Read straight from `summary.json` because per-cell CSV is wrong-base / correct-base split.

In [ ]:
import json
rows = []
for ds_label, ds_dir in [('PlotQA', 'experiment_e7_plotqa_full'), ('InfoVQA', 'experiment_e7_infographicvqa_full')]:
    base_dir = ROOT / 'outputs' / ds_dir
    for model_dir in sorted(base_dir.iterdir()):
        if not model_dir.is_dir() or model_dir.name == 'analysis':
            continue
        run_dirs = sorted([d for d in model_dir.iterdir() if d.is_dir()])
        if not run_dirs:
            continue
        sp = run_dirs[-1] / 'summary.json'
        if not sp.exists():
            continue
        s = json.load(open(sp))
        em_b = s.get('target_only', {}).get('accuracy_exact', 0)
        em_a = s.get('target_plus_irrelevant_number_S1', {}).get('accuracy_exact', 0)
        em_m = s.get('target_plus_irrelevant_number_masked_S1', {}).get('accuracy_exact', 0)
        em_d = s.get('target_plus_irrelevant_neutral', {}).get('accuracy_exact', 0)
        rows.append({'dataset': ds_label, 'model': model_dir.name,
                     'em_b': em_b, 'em_d': em_d, 'em_a': em_a, 'em_m': em_m,
                     'em_a_minus_b': em_a - em_b})
em = pd.DataFrame(rows).round(3)
print(em.sort_values(['dataset', 'em_a_minus_b'], ascending=[True, False]).to_string(index=False))

## 6. Cross-link

- `docs/insights/phase1-p0-v3-summary.md` — umbrella headline panel.
- `docs/insights/E5e-mathvista-reasoning-evidence.md` — H2-collapse via reasoning mode (different mechanism, same surface).
- `docs/insights/paper-section-7-4-mitigation-free-lunch.md` — §7.4.5 E6 mitigation that turns the PlotQA accuracy-positive baseline into a strict free-lunch.